# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [4]:
col = "Credit_Utilization_Ratio"

## <font color = 'skyblue'> ANÁLISIS GENERAL

El porcentaje de crédito utilizado pareciera no muy conclusivo: las medianas de Bad, Good y Standard parecen no significativamente diferentes

In [5]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [6]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [7]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Credit_Utilization_Ratio_Decile,,,,,,,,,,
0,20.000000,25.344974,10000,0.1,3043,2417,4540,0.3043,0.2417,0.4540
1,25.345261,27.177654,10000,0.1,2592,2726,4682,0.2592,0.2726,0.4682
2,27.178066,28.916763,10000,0.1,2352,2875,4773,0.2352,0.2875,0.4773
3,28.917108,30.618755,10000,0.1,2369,3020,4611,0.2369,0.3020,0.4611
4,30.619097,32.305677,10000,0.1,2343,3000,4657,0.2343,0.3000,0.4657
5,32.305891,33.980185,10000,0.1,2403,3026,4571,0.2403,0.3026,0.4571
6,33.980471,35.664972,10000,0.1,2390,3072,4538,0.2390,0.3072,0.4538
7,35.665094,37.317021,10000,0.1,2426,3033,4541,0.2426,0.3033,0.4541
8,37.317236,39.043323,10000,0.1,2336,3150,4514,0.2336,0.3150,0.4514


No missing values found.
No infinite values found.
No duplicate rows found.


In [10]:
df[(df[continuous_variable] >= 35.665094) & (df[continuous_variable] <= 37.317021) & (df['Credit_Score'] == 0)].shape

(2426, 85)

Proporción de Buenos: se observa una relación positiva la utilización del crédito y la proporción de deduores.

Proporción de standard: la realación entre la utilización del crédito es mixta: utilizaciones de crédito inferiores a 30% (aprox.) muestran una relación positiva, por encima de este umbral, la relación es negativa.

Proporción de malos: la proporción de malos y la utilización del crédito es, en tendencia, negativa, de modo que a mayor utilización de tarjetas menor proporción de malo, lo cual no es razonable.

In [11]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y la variable:

In [12]:
group_map = {0: "Group_1", 
             1: "Group_2", 
             2: "Group_2", 
             3: "Group_2", 
             4: "Group_2",
             5: "Group_2", 
             6: "Group_2",
             7: "Group_2",
             8: "Group_2",
             9: "Group_3"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Credit_Utilization_Ratio,,,,,,,,,,
Group_1,20.000000,25.344974,10000,0.1,3043,2417,4540,0.304300,0.241700,0.454000
Group_2,25.345261,39.043323,80000,0.8,19211,23902,36887,0.240138,0.298775,0.461087
Group_3,39.043436,50.000000,10000,0.1,1514,4065,4421,0.151400,0.406500,0.442100


No missing values found.
No infinite values found.
No duplicate rows found.


In [13]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [14]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [15]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Credit_Utilization_Ratio,"100,000.00",32.29,5.12,20.00,28.05,32.31,36.50,50.00


Todos los coeficientes son significativos.

Credit_Utilization_Ratio_Scaled 0.9949: el coeficiente es positivo: por cada punto porentual de utilización del crédito, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.7657: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.6970: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [16]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.056887
         Iterations: 11
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0569e+05
Model:                   OrderedModel   AIC:                         2.114e+05
Method:            Maximum Likelihood   BIC:                         2.114e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        13:10:05                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                      coe

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Credit_Utilization_Ratio_Decile  0.0525: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.9353: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.6956: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [18]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.057745
         Iterations: 9
         Function evaluations: 11
         Gradient evaluations: 11
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0577e+05
Model:                   OrderedModel   AIC:                         2.116e+05
Method:            Maximum Likelihood   BIC:                         2.116e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        13:12:45                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                      coef

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Credit_Utilization_Ratio 0.4063: Por cada grupo adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -0.3610: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 0.6979: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [19]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.056287
         Iterations: 12
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0563e+05
Model:                   OrderedModel   AIC:                         2.113e+05
Method:            Maximum Likelihood   BIC:                         2.113e+05
Date:                Sat, 05 Apr 2025                                         
Time:                        13:13:42                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                       co

## <font color = 'skyblue'> CONCLUSIONES

Esta variable no tiene un sentido lógico: sugiere que a mayor utilización del crédito menor la prorción de malos deudores, a la vez que mayor es la de buenos deudores. Esto no tiene mucho sentido.

## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
